In [ ]:
!pip install -q "numpy<2.3"
!pip install -q terratorch

In [ ]:
import numpy as np
import rasterio
import torch
from torch.utils.data import DataLoader, Dataset

In [ ]:
# Download the public image and mask folders from Google Drive.
# No Google Drive sign-in or mounting is required.
import os
import sys
import subprocess
import glob

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

DATASET_URL = "https://drive.google.com/drive/folders/1cvRcnRx9Kubj1rG3ntx2wN1GjjEmrd3N?usp=sharing"

# Keep the same folder layout expected by the rest of this notebook.
base_dir = "/content/Prithvi"
download_dir = f"{base_dir}/chips"
os.makedirs(download_dir, exist_ok=True)
os.makedirs(f"{base_dir}/model", exist_ok=True)

print("Downloading the public dataset from Google Drive...")
gdown.download_folder(
    url=DATASET_URL,
    output=download_dir,
    quiet=False,
    use_cookies=False,
    remaining_ok=True
)

# Locate the images/ and masks/ folders, including if Drive adds a parent folder.
image_folders = glob.glob(f"{download_dir}/**/images", recursive=True)
dataset_root = None
for folder in image_folders:
    parent = os.path.dirname(folder)
    if os.path.isdir(f"{parent}/masks"):
        dataset_root = parent
        break

if dataset_root is None:
    raise FileNotFoundError(
        "Could not find matching 'images' and 'masks' folders after download. "
        "Check that the Google Drive folder is shared as 'Anyone with the link – Viewer'."
    )

IMAGES_DIR = f"{dataset_root}/images"
MASKS_DIR = f"{dataset_root}/masks"

# Confirm that all nine image chips and their matching masks are present.
expected_files = []
for box in range(1, 10):
    expected_files.extend([
        f"{IMAGES_DIR}/box{box:02d}_image.tif",
        f"{MASKS_DIR}/box{box:02d}_mask.tif"
    ])
missing_files = [path for path in expected_files if not os.path.isfile(path)]
if missing_files:
    raise FileNotFoundError(
        "The dataset download is incomplete. Missing files:\n" + "\n".join(missing_files)
    )

print("\nDataset download and file check completed successfully.")
print(f"Images folder: {IMAGES_DIR}")
print(f"Masks folder : {MASKS_DIR}")
print("Found all 9 image chips and all 9 matching masks.")

TRAIN_BOXES = [1, 2, 3, 4, 5, 6, 7]
VAL_BOXES = [8, 9]

PRITHVI_VERSION = 'EO-2.0'
STATS = {
    # from the Prithvi-EO-1.0-100M config
    'EO-1.0-100M': {'mean': [775.229, 1080.993, 1228.586, 2497.202, 2204.214, 1610.832],
                    'std':  [1281.526, 1270.030, 1399.480, 1368.345, 1291.676, 1154.506]},
    # from the Prithvi-EO-2.0-300M-TL config
    'EO-2.0':      {'mean': [1087.0, 1342.0, 1433.0, 2734.0, 1958.0, 1363.0],
                    'std':  [2248.0, 2179.0, 2178.0, 1850.0, 1242.0, 1049.0]},
}

SCALE_FACTOR = 'auto'

ADD_TIME_AXIS = True

IGNORE_INDEX = 255
BATCH_SIZE = 2

CLASS_NAMES = {0: 'other', 1: 'raised bog', 2: 'cutover bog', 3: 'grassland', IGNORE_INDEX: 'ignore'}

In [ ]:
class ChipDataset(Dataset):
    """Loads box images + masks once, scales them for Prithvi, hands them out as tensors."""

    def __init__(self, box_ids, augment=False):
        self.box_ids = list(box_ids)
        self.augment = augment
        stats = STATS[PRITHVI_VERSION]
        self.mean = np.array(stats['mean'], dtype='float32').reshape(6, 1, 1)
        self.std = np.array(stats['std'], dtype='float32').reshape(6, 1, 1)

        # check every file is there before doing anything else
        wanted = []
        for box in self.box_ids:
            wanted += [f'{IMAGES_DIR}/box{box:02d}_image.tif', f'{MASKS_DIR}/box{box:02d}_mask.tif']
        absent = [p for p in wanted if not _exists(p)]
        if absent:
            raise FileNotFoundError('These files are missing (did you pull all these boxes?):\n  '
                                    + '\n  '.join(absent))

        # With only a handful of small chips we keep them all in memory.
        # (When you scale up to hundreds of chips, switch to loading each one
        #  from Drive only when it is asked for.)
        self.images, self.masks = [], []
        for box in self.box_ids:
            image, mask = self._load_chip(box)
            self.images.append(image)
            self.masks.append(mask)

    def _load_chip(self, box):
        tag = f'box{box:02d}'
        with rasterio.open(f'{IMAGES_DIR}/{tag}_image.tif') as src:
            image = src.read().astype('float32')                  # (6, 224, 224)
            nodata, img_transform, img_crs = src.nodata, src.transform, src.crs
        with rasterio.open(f'{MASKS_DIR}/{tag}_mask.tif') as src:
            mask = src.read(1).astype('int64')                    # (224, 224)
            msk_transform, msk_crs = src.transform, src.crs

        if image.shape[0] != 6:
            raise ValueError(f'{tag}: expected 6 bands, found {image.shape[0]}')
        if image.shape[1:] != mask.shape:
            raise ValueError(f'{tag}: image is {image.shape[1:]} but mask is {mask.shape}')

        # are the two files really on the same grid?
        shift = max(abs(a - b) for a, b in zip(tuple(img_transform)[:6], tuple(msk_transform)[:6]))
        grid_ok = (img_crs == msk_crs) and shift < 1.0            # less than 1 metre apart
        if not grid_ok:
            print(f'  WARNING {tag}: image and mask do not seem to sit on the same grid '
                  f'(origin/pixel-size differ by {shift:.1f} m, or the map projections differ)')

        # "no data" can be written three ways: empty (NaN), a special number, or zeros
        missing = np.isnan(image).any(axis=0)
        if nodata is not None and not np.isnan(nodata) and nodata != 0:
            missing |= (image == nodata).any(axis=0)
        missing |= (image == 0).all(axis=0)

        valid = image[:, ~missing]
        median = float(np.median(valid)) if valid.size else float('nan')
        if SCALE_FACTOR == 'auto':
            scale = 10000.0 if (valid.size and median < 10) else 1.0
        else:
            scale = float(SCALE_FACTOR)
        image = image * scale

        # empty pixels: neutral filler in the image, "ignore" in the mask
        image = np.where(missing[None], self.mean, image)
        mask[missing] = IGNORE_INDEX

        # scale the way Prithvi was trained
        image = ((image - self.mean) / self.std).astype('float32')

        if not valid.size:
            note = 'NO usable pixels at all - consider dropping this box'
        elif scale != 1.0:
            note = f'median value {median:.3f} -> looks like 0-1 reflectance, multiplied by {scale:g}'
        else:
            note = f'median value {median:.0f} -> raw-style reflectance, left as is'
        print(f'  {tag}: {note} | {int(missing.sum())} empty pixels'
              f'{"" if grid_ok else " | GRID WARNING"}')

        odd = np.setdiff1d(np.unique(mask[~missing]), [0, 1, 2, 3])
        if odd.size:
            print(f'  WARNING {tag}: mask contains unexpected values {odd.tolist()}')
        return image, mask

    def __len__(self):
        return len(self.box_ids)

    def __getitem__(self, idx):
        image, mask = self.images[idx], self.masks[idx]
        if self.augment:
            image, mask = self._augment(image, mask)
        image = torch.from_numpy(np.ascontiguousarray(image))     # (6, 224, 224)
        mask = torch.from_numpy(np.ascontiguousarray(mask))       # (224, 224)
        if ADD_TIME_AXIS:
            image = image.unsqueeze(1)                            # (6, 1, 224, 224)
        return image, mask

    @staticmethod
    def _augment(image, mask):
        """Random flips and 90-degree turns, applied the SAME way to image and
        mask so they stay lined up. Only used on the training set."""
        if np.random.rand() < 0.5:                                # left-right flip
            image, mask = image[:, :, ::-1], mask[:, ::-1]
        if np.random.rand() < 0.5:                                # up-down flip
            image, mask = image[:, ::-1, :], mask[::-1, :]
        turns = np.random.randint(4)                              # 0, 90, 180 or 270 degrees
        if turns:
            image = np.rot90(image, turns, axes=(1, 2))
            mask = np.rot90(mask, turns, axes=(0, 1))
        return image, mask


def _exists(path):
    import os
    return os.path.exists(path)



In [ ]:
train_set = ChipDataset(TRAIN_BOXES, augment=True)
val_set = ChipDataset(VAL_BOXES, augment=False)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
# ---------------------------------------------------------------------
# STEP 4: SANITY CHECK -- WHAT WILL THE MODEL ACTUALLY RECEIVE?
# ---------------------------------------------------------------------
def class_breakdown(dataset):
    counts = {k: sum(int((m == k).sum()) for m in dataset.masks) for k in CLASS_NAMES}
    total = sum(counts.values())
    return '\n'.join(f'    {CLASS_NAMES[k]:24s} {c:>8,} pixels  ({c / total:.1%})'
                     for k, c in counts.items())


print(f'\nTrain: {len(train_set)} boxes  |  Validation: {len(val_set)} boxes')
print('\nWhat is in the training masks:')
print(class_breakdown(train_set))
print('What is in the validation masks:')
print(class_breakdown(val_set))

images, masks = next(iter(train_loader))
layout = '(batch, bands, time, height, width)' if ADD_TIME_AXIS else '(batch, bands, height, width)'
print(f'\nOne batch of training data:')
print(f'  images: {tuple(images.shape)}  {layout}  {images.dtype}')
print(f'  masks : {tuple(masks.shape)}  (batch, height, width)  {masks.dtype}')
print(f'  image values run from {images.min():.2f} to {images.max():.2f} '
      f'(after scaling, most values should sit roughly between -3 and +5)')
print(f'  mask values present: {sorted(torch.unique(masks).tolist())}  '
      f'(0-3 = your classes, {IGNORE_INDEX} = ignore)')


In [ ]:
import torch
from terratorch.datasets import HLSBands
from terratorch.models import EncoderDecoderFactory

In [ ]:
NUM_CLASSES = 4
HEAD_DROPOUT = 0.1

BANDS = [HLSBands.BLUE, HLSBands.GREEN, HLSBands.RED, HLSBands.NIR_NARROW, HLSBands.SWIR_1, HLSBands.SWIR_2]

In [ ]:
factory = EncoderDecoderFactory()

model = factory.build_model(
    task="segmentation",
    backbone="prithvi_eo_v2_300",
    backbone_pretrained=True,
    backbone_bands=BANDS,
    backbone_num_frames=1,
    necks=[
        {"name": "SelectIndices", "indices": [5, 11, 17, 23]},    # 4 depths through the network
        {"name": "ReshapeTokensToImage"},                          # tokens -> small pictures
        {"name": "LearnedInterpolateToPyramidal"},                 # small pictures -> a real pyramid
    ],
    decoder="UNetDecoder",
    decoder_channels=[512, 256, 128, 64],
    num_classes=NUM_CLASSES,
    head_dropout=HEAD_DROPOUT,
)

# Freeze Prithvi itself; only the decoder + head (built above) will train.
model.freeze_encoder()

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model built: {n_total:,} parameters in total')
print(f'  trainable (decoder + head): {n_trainable:,}  ({n_trainable / n_total:.1%})')
print(f'  frozen (Prithvi backbone) : {n_total - n_trainable:,}  ({1 - n_trainable / n_total:.1%})')



In [ ]:
# ---------------------------------------------------------------------
# SANITY CHECK: RUN ONE REAL BATCH FROM build_dataset.py THROUGH IT
# ---------------------------------------------------------------------
assert 'train_loader' in globals(), (
    'train_loader is not defined - run build_dataset.py in this same notebook first.')

images, masks = next(iter(train_loader))

model.eval()
with torch.no_grad():
    result = model(images)
logits = result.output    # model(...) returns an object; .output is the actual tensor

print(f'\nInput batch  : {tuple(images.shape)}')
print(f'Output logits: {tuple(logits.shape)}  (batch, {NUM_CLASSES} classes, height, width)')
expected = (images.shape[0], NUM_CLASSES, 224, 224)
assert tuple(logits.shape) == expected, f'expected {expected}, got {tuple(logits.shape)}'
print('Shape check passed: one score per class, for every pixel, for every image in the batch.')

guess = logits.argmax(dim=1)      # the model's current best guess per pixel
print(f'\nBefore any training this guess is meaningless - classes present in it right now: '
      f'{sorted(torch.unique(guess).tolist())}. That becomes useful once we train it next.')



In [ ]:
import os
import torch.nn as nn

In [ ]:
needed = ['model', 'train_loader', 'val_loader', 'train_set', 'IGNORE_INDEX', 'NUM_CLASSES', "CLASS_NAMES", 'base_dir']

missing = [n for n in needed if n not in globals()]

if missing:
  raise NameError(f'{missing} not defined')

In [ ]:
NUM_EPOCHS = 100
PATIENCE = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.1
CHECKPOINT_PATH = f'{base_dir}/model/best_decoder'

torch.manual_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = model.to(device)


In [ ]:
# ---------------------------------------------------------------------
# STEP 1: WEIGH THE RARE CLASSES UP
# ---------------------------------------------------------------------
all_train_pixels = np.concatenate([m.ravel() for m in train_set.masks])
valid_pixels = all_train_pixels[all_train_pixels != IGNORE_INDEX]
counts = np.bincount(valid_pixels, minlength=NUM_CLASSES)
if (counts == 0).any():
    print('WARNING: at least one class has 0 pixels in the training chips - the model '
         'cannot learn that class from this split. Consider a different TRAIN_BOXES split.')
class_weights = counts.sum() / (NUM_CLASSES * np.maximum(counts, 1))
class_weights = torch.tensor(class_weights, dtype=torch.float32)

print('Class weights (higher = counts for more when grading the model):')
for c in range(NUM_CLASSES):
    print(f'  {CLASS_NAMES[c]:14s} {counts[c]:>8,} pixels  ->  weight {class_weights[c]:.2f}')

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device), ignore_index=IGNORE_INDEX)


In [ ]:
# ---------------------------------------------------------------------
# STEP 2: OPTIMIZER (ONLY THE TRAINABLE PARAMETERS) + A SHRINKING LEARNING RATE
# ---------------------------------------------------------------------
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5)


In [ ]:
# ---------------------------------------------------------------------
# STEP 3: ONE PASS OVER A LOADER, SHARED BY TRAINING AND VALIDATION
# ---------------------------------------------------------------------
def run_one_epoch(loader, train):
    model.train(train)
    if train:
        model.encoder.eval()      # keep the frozen backbone deterministic even in train mode
    total_loss, correct, total = 0.0, 0, 0
    correct_per_class = np.zeros(NUM_CLASSES)
    seen_per_class = np.zeros(NUM_CLASSES)
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        with torch.set_grad_enabled(train):
            logits = model(images).output
            loss = criterion(logits, masks)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * images.size(0)
        valid = masks != IGNORE_INDEX
        pred = logits.argmax(1)
        correct += ((pred == masks) & valid).sum().item()
        total += valid.sum().item()
        for c in range(NUM_CLASSES):
            is_c = masks == c
            seen_per_class[c] += is_c.sum().item()
            correct_per_class[c] += ((pred == c) & is_c).sum().item()
    recall = np.divide(correct_per_class, seen_per_class,
                       out=np.full(NUM_CLASSES, np.nan), where=seen_per_class > 0)
    return total_loss / len(loader.dataset), (correct / total if total else float('nan')), recall


In [ ]:
# ---------------------------------------------------------------------
# STEP 4: THE TRAINING LOOP
# ---------------------------------------------------------------------
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)

best_val_loss = float('inf')
epochs_without_improvement = 0
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc, _ = run_one_epoch(train_loader, train=True)
    val_loss, val_acc, val_recall = run_one_epoch(val_loader, train=False)
    scheduler.step(val_loss)
    history.append(dict(epoch=epoch, train_loss=train_loss, val_loss=val_loss,
                        train_acc=train_acc, val_acc=val_acc, val_recall=val_recall.tolist()))

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        epochs_without_improvement = 0
        # only the parts that actually trained - Prithvi's 300M weights never change,
        # so re-saving them every time would just burn through your Drive quota
        trainable_state = {k: v for k, v in model.state_dict().items() if not k.startswith('encoder.')}
        torch.save(trainable_state, CHECKPOINT_PATH)
    else:
        epochs_without_improvement += 1

    flag = '  <- best so far, saved' if improved else ''
    recall_str = '  '.join(f'{n}={r:.0%}' if not np.isnan(r) else f'{n}=n/a'
                           for n, r in zip(CLASS_NAMES.values(), val_recall))
    print(f'epoch {epoch:3d}/{NUM_EPOCHS}  '
          f'train loss {train_loss:.3f} (acc {train_acc:.1%})  |  '
          f'val loss {val_loss:.3f} (acc {val_acc:.1%}){flag}')
    print(f'            val recall by class: {recall_str}')

    if epochs_without_improvement >= PATIENCE:
        print(f'\nNo improvement for {PATIENCE} epochs in a row - stopping early.')
        break

print(f'\nBest validation loss: {best_val_loss:.3f}')
print(f'Best decoder weights saved to: {CHECKPOINT_PATH}')

model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device), strict=False)
model.eval()
print('Reloaded the best checkpoint into `model` (Prithvi itself was never touched, '
     'so only the trained parts needed reloading). Ready for evaluation.')


In [ ]:
# ---------------------------------------------------------------------
# STEP 5: PLOT THE TRAINING CURVE, SO OVERFITTING IS EASY TO SPOT
# ---------------------------------------------------------------------
try:
    import matplotlib.pyplot as plt
    epochs_x = [h['epoch'] for h in history]
    plt.figure(figsize=(7, 4))
    plt.plot(epochs_x, [h['train_loss'] for h in history], label='train loss')
    plt.plot(epochs_x, [h['val_loss'] for h in history], label='val loss')
    plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title('Training curve')
    plt.tight_layout()
    curve_path = f'{base_dir}/model/training_curve.png'
    plt.savefig(curve_path, dpi=120)
    plt.show()
    print(f'Training curve saved -> {curve_path}')
    print('If val loss flattens out or rises while train loss keeps falling, that\'s '
         'overfitting - a real risk with only 5-7 training chips.')
except Exception as e:
    print(f'(Training curve skipped: {e})')

